In [ ]:
#!/usr/bin/env python3
"""
vigenere_tool.py - FAST educational template for attacking Vigenère ciphers.

Fixed: Simplified attack to use BEST shift per column only (no Cartesian explosion).
Added: Progress indicators, timing, and clearer output.

Features:
- clean_text: removes non-letters
- prime_factors: helper to factorize numbers
- kasiski_examination: find repeated sequences, distances, and common prime factors
- index_of_coincidence: compute IC for any text
- friedman_estimate: full Friedman approximations for key length
- suggest_key_lengths: combines Kasiski factors and Friedman estimates
- average_ic_by_keylen: compute average IC for subgroups by key length
- split_into_columns: organize text into columns for given key length
- chi_squared_for_shift: compute chi-square score for a single shift
- find_best_shifts: find BEST shift per column using chi-square (fast)
- key_from_shifts: convert shift list to key string
- decrypt_vigenere: decrypt ciphertext with a key
- attack_key_length: attack ONE key length with chi-square (returns best key)
- example_run: runs full pipeline with timing and clear output

Usage:
    python3 vigenere_tool.py          # runs example in <30 seconds
    import vigenere_tool as vt       # use functions in scripts

Notes:
- Kasiski: Factors distances to suggest k (common factors = likely key lengths).
- Friedman: Statistical estimate from overall IC deviation.
- Columns: text[i::k] for i=0..k-1 (cosets).
- Chi-square: Tests 26 shifts per column; picks lowest χ² (best English match).
- Attack: Uses BEST shift per column → one candidate key per key length.
"""

from collections import defaultdict, Counter
import itertools
import math
import time

ENGLISH_FREQ = {
    'A': 0.08167, 'B': 0.01492, 'C': 0.02782, 'D': 0.04253, 'E': 0.12702,
    'F': 0.02228, 'G': 0.02015, 'H': 0.06094, 'I': 0.06966, 'J': 0.00153,
    'K': 0.00772, 'L': 0.04025, 'M': 0.02406, 'N': 0.06749, 'O': 0.07507,
    'P': 0.01929, 'Q': 0.00095, 'R': 0.05987, 'S': 0.06327, 'T': 0.09056,
    'U': 0.02758, 'V': 0.00978, 'W': 0.02360, 'X': 0.00150, 'Y': 0.01974,
    'Z': 0.00074
}

#add your digrams, and ngrams!

ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def clean_text(s):
    """Remove non-letters and uppercase."""
    return ''.join(ch for ch in s.upper() if ch.isalpha())

def prime_factors(n):
    """Return list of prime factors of n (with duplicates if multiple)."""
    if n <= 1:
        return []
    factors = []
    while n % 2 == 0:
        factors.append(2)
        n //= 2
    i = 3
    while i * i <= n:
        while n % i == 0:
            factors.append(i)
            n //= i
        i += 2
    if n > 1:
        factors.append(n)
    return factors

def kasiski_examination(text, min_len=3, max_len=4):  # Reduced max_len for speed
    """
    Kasiski examination: Find repeated sequences, compute distances, count prime factors.
    """
    print("  Running Kasiski examination...")
    seq_positions = defaultdict(list)
    for L in range(min_len, max_len + 1):
        for i in range(len(text) - L + 1):
            seq = text[i:i + L]
            seq_positions[seq].append(i)
    
    repeats = {seq: pos for seq, pos in seq_positions.items() if len(pos) > 1}
    distances = defaultdict(list)
    for seq, pos in repeats.items():
        for i in range(1, len(pos)):
            distances[seq].append(pos[i] - pos[i - 1])
    
    factor_count = Counter()
    for dlist in distances.values():
        for d in dlist:
            factors = prime_factors(d)
            for f in set(factors):
                factor_count[f] += 1
    
    return repeats, distances, factor_count.most_common(10)

def index_of_coincidence(text):
    """IC = sum f(f-1) / N(N-1) for letter frequencies f."""
    N = len(text)
    if N <= 1:
        return 0.0
    freqs = Counter(text)
    numerator = sum(v * (v - 1) for v in freqs.values())
    denom = N * (N - 1)
    return numerator / denom

def friedman_estimate(text, ic_plain=0.0667, ic_rand=0.0385):
    """Full Friedman test: simple + corrected key length estimates."""
    N = len(text)
    IC_obs = index_of_coincidence(text)
    num = ic_plain - ic_rand
    den_simple = IC_obs - ic_rand
    K_simple = num / den_simple if den_simple != 0 else float('inf')
    
    den_corr = (N - 1) * IC_obs - ic_rand * N + ic_plain
    K_corr = (num * N) / den_corr if den_corr != 0 else float('inf')
    
    return {'IC': IC_obs, 'N': N, 'K_simple': K_simple, 'K_corrected': K_corr}

def suggest_key_lengths(factor_common, fried, min_k=2, max_k=15, top_n_kas=5):
    """Suggest candidates from Kasiski factors and Friedman estimates."""
    from_kas = [f for f, c in factor_common[:top_n_kas] if min_k <= f <= max_k]
    from_fried = []
    for est in [fried['K_simple'], fried['K_corrected']]:
        if math.isfinite(est):
            for offset in [-1, 0, 1]:
                cand = round(est + offset)
                if min_k <= cand <= max_k:
                    from_fried.append(cand)
    suggested = sorted(set(from_kas + from_fried))
    return suggested if suggested else list(range(min_k, min(max_k + 1, 10)))

def average_ic_by_keylen(text, max_k=15):
    """Compute average IC for subgroups by key length k=1 to max_k."""
    results = []
    for k in range(1, max_k + 1):
        columns = [text[i::k] for i in range(k)]
        ics = [index_of_coincidence(col) for col in columns if len(col) > 1]
        avg_ic = sum(ics) / len(ics) if ics else 0.0
        results.append((k, avg_ic, ics))
    return results

def split_into_columns(text, k):
    """Split text into k columns: column i = text[i::k]."""
    return [text[i::k] for i in range(k)]

def chi_squared_for_shift(column, shift, eng_freq=ENGLISH_FREQ):
    """Chi-square for column decrypted with backward shift 'shift'."""
    N = len(column)
    if N == 0:
        return float('inf'), {}, {}
    
    observed = {ch: 0 for ch in ALPHABET}
    for c in column:
        c_idx = ALPHABET.index(c)
        p_idx = (c_idx - shift) % 26
        p = ALPHABET[p_idx]
        observed[p] += 1
    
    expected = {ch: eng_freq[ch] * N for ch in ALPHABET}
    chi = 0.0
    for ch in ALPHABET:
        o = observed[ch]
        e = expected[ch]
        if e > 0:
            chi += (o - e) ** 2 / e
    
    return chi, observed, expected

def find_best_shifts(text, keylen):
    """
    Find BEST shift per column using chi-square.
    Returns: shifts_info [(shift, chi) for each column], best_shifts [list of best per column]
    """
    print(f"    Computing chi-square for {keylen} columns...")
    columns = split_into_columns(text, keylen)
    shifts_info = []
    best_shifts = []
    
    for col_idx, col in enumerate(columns):
        print(f"      Column {col_idx+1}/{keylen} ({len(col)} letters)...", end=' ')
        col_shifts = []
        for shift in range(26):
            chi, _, _ = chi_squared_for_shift(col, shift)
            col_shifts.append((shift, chi))
        col_shifts.sort(key=lambda x: x[1])
        shifts_info.append(col_shifts)
        best_shifts.append(col_shifts[0][0])
        print(f"best shift={col_shifts[0][0]}, χ²={col_shifts[0][1]:.1f}")
    
    return shifts_info, best_shifts

def key_from_shifts(shifts):
    """Convert shift numbers (0-25) to key string (shifts ARE key offsets)."""
    return ''.join(ALPHABET[s] for s in shifts)

def decrypt_vigenere(ciphertext, key):
    """Decrypt Vigenère: plaintext = ciphertext - key (repeating)."""
    plaintext = []
    key = clean_text(key)
    key_len = len(key)
    key_idx = 0
    
    for ch in ciphertext:
        if ch.isalpha():
            c_idx = ALPHABET.index(ch.upper())
            k_idx = ALPHABET.index(key[key_idx % key_len])
            p_idx = (c_idx - k_idx) % 26
            p = ALPHABET[p_idx] if ch.isupper() else ALPHABET[p_idx].lower()
            plaintext.append(p)
            key_idx += 1
        else:
            plaintext.append(ch)
    return ''.join(plaintext)

def attack_key_length(text, keylen):
    """
    Attack ONE key length: find best shifts, derive key, decrypt, score result.
    Returns: dict with key, shifts, chi_scores, plaintext, score
    """
    shifts_info, best_shifts = find_best_shifts(text, keylen)
    key = key_from_shifts(best_shifts)
    plaintext = decrypt_vigenere(text, key)
    
    # Simple scoring: count English-like patterns
    pt_upper = plaintext.upper()
    score = (pt_upper.count(' THE ') + pt_upper.count(' AND ') + pt_upper.count(' OF ') +
             pt_upper.count(' TO ') + pt_upper.count(' IN ') + pt_upper.count(' IS ') +
             pt_upper.count(' THAT ') + pt_upper.count(' WITH ') + pt_upper.count(' FROM '))
    
    chi_scores = [info[0][1] for info in shifts_info]
    
    return {
        'key': key,
        'shifts': best_shifts,
        'chi_scores': chi_scores,
        'plaintext': plaintext,
        'score': score,
        'snippet': plaintext[:200]
    }

def example_run():
    print("=== Vigenère Attack Tool - Educational Demo ===\n")
    
    sample_ct = """PVWUL UWRAS GIITA YKENW YRWOG KFJTA KDSSM OEJLN KEXIT RWMGN XVWIG NLQAG NZWTH XPVEO KIIDT YKLEV KEXRT RWMGN XVSFV NIMSM ORRIM ESSRG GGTRH DZQAM KCCTP UKLON YRRDR KRVST MFMNK UDENH ITYPB KUNUW KRLIL RZJET TUXET IYMNZ YYEVX YYEPX JESTH TCCAF GASRP UIPDK KCMGB UEFUM GCWOB TWPUX TTIDP KJXEK TDSRT RVXHB IRPAG JGLIE UJSPA OTELM XRHIM OFRSA OJXOK OTELV UEXEQ ZRRDX GIPYE OWIJX YLWWT YSSRG OEFEM NCIHX SKLON MYLEP GJVAB YVHIG TRDAK KKLAL SRPLM UNRIG MRPIE KVXHX CFVLW NVAAL HFVNB TKSWT YFREH LGSLB ZZGAE ZLVMH OCVEE OXMON YVBPX IKETB UEENW YFGIT RJXRT ZZJIV GKMOG XFQAG XLPEB SGSSX JKEXX YRRDT AKLOK OKCAG JDENR PVAIL NGIOI RVPOG MVFHX RQELY ZEHTY RZIHX NLOPU LPDKK JXOKK ZWRTK CWFHX KYNXY RRDYA CJIER RRCBK EXPKU GLEVO VWIGZ FXHBY VRVBX FRMXT KNELA JFEZG ELILS ZRILZ ICAKU LRDMN VEGXU WXHBX KCAYZ VVBXO EKBTV KMZXJ SCJHN EXHXH RTTBY K"""
    
    start_time = time.time()
    text = clean_text(sample_ct)
    print(f"✓ Cleaned ciphertext: {len(text)} letters")
    
    # 1. KASISKI EXAMINATION
    print(f"\n1. KASISKI EXAMINATION")
    repeats, distances, factor_common = kasiski_examination(text)
    print(f"  Found {len(repeats)} repeated sequences")
    print(f"  Top 5 repeats:")
    for seq, pos in sorted(repeats.items(), key=lambda x: len(x[1]), reverse=True)[:5]:
        print(f"    {seq}: positions {pos}")
    print(f"  Most common distance factors: {factor_common[:5]}")
    
    # 2. FRIEDMAN ESTIMATE
    print(f"\n2. FRIEDMAN KEY LENGTH ESTIMATE")
    fried = friedman_estimate(text)
    print(f"  Text IC: {fried['IC']:.5f} (English=0.0667, Random=0.0385)")
    print(f"  Simple estimate:  K ≈ {fried['K_simple']:.2f}")
    print(f"  Corrected estimate: K ≈ {fried['K_corrected']:.2f}")
    
    # 3. SUGGEST CANDIDATES
    suggested_ks = suggest_key_lengths(factor_common, fried)
    print(f"\n3. SUGGESTED KEY LENGTHS: {suggested_ks}")
    
    # 4. AVERAGE IC BY KEY LENGTH
    print(f"\n4. SUBGROUP IC ANALYSIS (for top candidates)")
    avg_ics = average_ic_by_keylen(text, max_k=max(suggested_ks))
    print(f"{'k':>2} {'Avg IC':>8} {'(English)':>10} {'Verdict'}")
    print("-" * 35)
    for k, avg_ic, _ in avg_ics:
        if k in suggested_ks:
            verdict = "★ HIGH" if avg_ic > 0.05 else "low"
            print(f"{k:>2} {avg_ic:>8.5f} {0.0667:>10.5f} {verdict:>8}")
    
    # 5. CHI-SQUARE ATTACK ON CANDIDATES
    print(f"\n5. CHI-SQUARE ATTACK ON CANDIDATE KEY LENGTHS")
    print(f"{'k':>2} {'Key':>6} {'Shifts':>15} {'χ² scores':>15} {'Score':>6} {'Snippet'}")
    print("-" * 80)
    
    attack_results = {}
    for k in suggested_ks:
        print(f"\n  Attacking key length {k}...")
        result = attack_key_length(text, k)
        attack_results[k] = result
        
        shifts_str = ",".join(f"{s:2d}" for s in result['shifts'])
        chi_str = f"[{min(result['chi_scores']):.0f}...{max(result['chi_scores']):.0f}]"
        
        print(f"  {k:>2} {result['key']:>6} {shifts_str:>15} {chi_str:>15} {result['score']:>6} | {result['snippet'][:60]}...")
    
    # 6. BEST RESULT
    print(f"\n6. BEST CANDIDATE:")
    best_k = max(attack_results.keys(), key=lambda k: attack_results[k]['score'])
    best_result = attack_results[best_k]
    print(f"  Key length {best_k}: Key = '{best_result['key']}'")
    print(f"  Score: {best_result['score']} (higher = more English-like)")
    print(f"  Decrypted: {best_result['snippet'][:300]}...")
    
    # 7. TIMING
    elapsed = time.time() - start_time
    print(f"\n✓ Analysis complete in {elapsed:.1f} seconds!")
    
    # Write full results to file
    out_lines = [
        f"VIGENERE ATTACK RESULTS - {time.strftime('%Y-%m-%d %H:%M:%S')}",
        f"Ciphertext length: {len(text)}",
        f"Kasiski top factors: {factor_common[:5]}",
        f"Friedman estimates: simple={fried['K_simple']:.2f}, corrected={fried['K_corrected']:.2f}",
        f"Suggested key lengths: {suggested_ks}",
        "",
        "CHI-SQUARE ATTACK RESULTS:",
        f"{'k':>2} {'Key':>6} {'Shifts':>20} {'χ² range':>12} {'Score':>6}",
        "-" * 55
    ]
    
    for k in sorted(attack_results):
        result = attack_results[k]
        shifts_str = ",".join(f"{s:2d}" for s in result['shifts'])
        chi_range = f"[{min(result['chi_scores']):.0f}..{max(result['chi_scores']):.0f}]"
        out_lines.append(f"{k:>2} {result['key']:>6} {shifts_str:>20} {chi_range:>12} {result['score']:>6}")
        out_lines.append(f"Plaintext: {result['plaintext'][:500]}...")
        out_lines.append("")
    
    out_file = 'vigenere_results.txt'
    with open(out_file, 'w') as f:
        f.write('\n'.join(out_lines))
    print(f"\n✓ Full results saved to {out_file}")

if __name__ == '__main__':
    example_run()

=== Vigenère Attack Tool - Educational Demo ===

✓ Cleaned ciphertext: 691 letters

1. KASISKI EXAMINATION
  Running Kasiski examination...
  Found 65 repeated sequences
  Top 5 repeats:
    KEX: positions [40, 80, 481, 584]
    RRD: positions [131, 296, 486, 570]
    KLO: positions [126, 331, 491]
    YRR: positions [130, 485, 569]
    XHB: positions [247, 601, 651]
  Most common distance factors: [(2, 46), (5, 45), (3, 20), (7, 13), (17, 8)]

2. FRIEDMAN KEY LENGTH ESTIMATE
  Text IC: 0.04299 (English=0.0667, Random=0.0385)
  Simple estimate:  K ≈ 6.28
  Corrected estimate: K ≈ 6.24

3. SUGGESTED KEY LENGTHS: [2, 3, 5, 6, 7]

4. SUBGROUP IC ANALYSIS (for top candidates)
 k   Avg IC  (English) Verdict
-----------------------------------
 2  0.04375    0.06670      low
 3  0.04212    0.06670      low
 5  0.05393    0.06670   ★ HIGH
 6  0.04249    0.06670      low
 7  0.04294    0.06670      low

5. CHI-SQUARE ATTACK ON CANDIDATE KEY LENGTHS
 k    Key          Shifts       χ² scores  Sc